In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#Binoculars

##Imports For Binoculars

In [ ]:
# Setup project directories
import os
PROJECT_DIR = "/content/drive/MyDrive/BEEMO_Reproducibility"
DIRS = {
    'data': os.path.join(PROJECT_DIR, 'data'),
    'results': os.path.join(PROJECT_DIR, 'results'),
    'logs': os.path.join(PROJECT_DIR, 'logs'),
    'figures': os.path.join(PROJECT_DIR, 'figures'),
}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)
print(f" Project dir: {PROJECT_DIR}")

print("\n Installing dependencies...")
!pip uninstall -y transformers tokenizers 2>/dev/null
!pip install --upgrade pip setuptools wheel

# Install pre-built tokenizers FIRST
!pip install --only-binary=:all: tokenizers

# Install compatible transformers
!pip install transformers accelerate datasets sentencepiece scikit-learn tqdm torch

#  Clone Binoculars
!rm -rf Binoculars
!git clone https://github.com/ahans30/Binoculars.git

import sys
sys.path.insert(0, '/content/Binoculars')
from binoculars import Binoculars
print("\n Binoculars imported ")

 Project dir: /content/drive/MyDrive/BEEMO_Reproducibility

 Installing dependencies...
Found existing installation: transformers 4.57.2
Uninstalling transformers-4.57.2:
  Successfully uninstalled transformers-4.57.2
Found existing installation: tokenizers 0.22.1
Uninstalling tokenizers-0.22.1:
  Successfully uninstalled tokenizers-0.22.1
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Using cached transformers-4.57.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.2-py3-none-any.whl (12.0 MB)
Cloning into 'Binoculars'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 208 (delta 72), reused 64 (delta 64), pack-reused 119 (from 1)
Receiving objects: 100% (208/208), 54.08 MiB | 15.04 MiB/s, done.
Resolving deltas:

In [ ]:
import os
import pandas as pd
import ast
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import torch
from binoculars import Binoculars

##BEEMO Dataset Loading

In [ ]:
# Load BEEMO Dataset

print(" Loading BEEMO dataset from HuggingFace...")
beemo_dataset = load_dataset("toloka/beemo")

# Explore the dataset
print(f"\n Dataset structure:")
print(beemo_dataset)

# Convert to pandas for easier manipulation
df = beemo_dataset['train'].to_pandas()

print(f"\n Dataset size: {len(df)} samples")
print(f"\n Columns: {df.columns.tolist()}")
print(f"\n Categories: {df['category'].value_counts()}")
print(f"\n Models: {df['model'].value_counts()}")

# Save raw dataset info
df.to_csv(os.path.join(DIRS['data'], 'beemo_full.csv'), index=False)
print(f"\n Saved dataset to {DIRS['data']}/beemo_full.csv")

 Loading BEEMO dataset from HuggingFace...

 Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'category', 'model', 'prompt_id', 'prompt', 'model_output', 'human_output', 'human_edits', 'llama-3.1-70b_edits', 'gpt-4o_edits'],
        num_rows: 2187
    })
})

 Dataset size: 2187 samples

 Columns: ['id', 'category', 'model', 'prompt_id', 'prompt', 'model_output', 'human_output', 'human_edits', 'llama-3.1-70b_edits', 'gpt-4o_edits']

 Categories: category
Generation    585
Rewrite       525
Open QA       483
Summarize     389
Closed QA     205
Name: count, dtype: int64

 Models: model
meta-llama/Llama-2-13b-chat-hf          234
meta-llama/Llama-2-7b-chat-hf           234
HuggingFaceH4/zephyr-7b-beta            229
mistralai/Mixtral-8x7B-Instruct-v0.1    226
meta-llama/Llama-2-70b-chat-hf          224
allenai/tulu-2-7b                       215
allenai/tulu-2-13b                      211
google/gemma-7b-it                      209
google/gemma-2b-it          

##Binoculars Implementation

In [ ]:
# Initialize Binoculars Detector

print("\n Initializing Binoculars ")
binoculars = Binoculars()
print(" Binoculars ready!")

# Test on a sample
test_text = df.iloc[0]['model_output']
test_score = binoculars.compute_score(test_text)
print(f"\n Test score: {test_score:.4f}")



 Initializing Binoculars 


tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

 Binoculars ready!

 Test score: 0.8708


In [ ]:
def prepare_comparison_data(df, text1_col, text2_col, label1=0, label2=1):
    texts = []
    labels = []

    for _, row in df.iterrows():
        val1 = row.get(text1_col, None)
        if pd.notna(val1):
            if isinstance(val1, str) and val1.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val1)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        texts.append(text)
                                        labels.append(label1)
                except:
                    text = str(val1).strip()
                    if text:
                        texts.append(text)
                        labels.append(label1)
            else:
                text = str(val1).strip()
                if text:
                    texts.append(text)
                    labels.append(label1)

        val2 = row.get(text2_col, None)
        if pd.notna(val2):
            if isinstance(val2, str) and val2.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val2)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        texts.append(text)
                                        labels.append(label2)

                except:
                    text = str(val2).strip()
                    if text:
                        texts.append(text)
                        labels.append(label2)
            else:
                text = str(val2).strip()
                if text:
                    texts.append(text)
                    labels.append(label2)

    return texts, labels

# Define the 7 comparison tasks
comparison_tasks = {
    'H_vs_M': ('human_output', 'model_output', 'Human vs Machine-generated'),
    'E_vs_M': ('human_edits', 'model_output', 'Expert-edited vs Machine-generated'),
    'H_vs_E': ('human_output', 'human_edits', 'Human vs Expert-edited'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output', 'Llama-edited vs Machine-generated'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits', 'Llama-edited vs Human'),
    'G_vs_M': ('gpt-4o_edits', 'model_output', 'GPT-4o-edited vs Machine-generated'),
    'H_vs_G': ('human_output', 'gpt-4o_edits', 'Human vs GPT-4o-edited'),
}

print("\n 7 comparison tasks:")
for task_name, (col1, col2, description) in comparison_tasks.items():
    print(f"  • {task_name}: {description}")



 7 comparison tasks:
  • H_vs_M: Human vs Machine-generated
  • E_vs_M: Expert-edited vs Machine-generated
  • H_vs_E: Human vs Expert-edited
  • L_vs_M: Llama-edited vs Machine-generated
  • H_vs_L: Llama-edited vs Human
  • G_vs_M: GPT-4o-edited vs Machine-generated
  • H_vs_G: Human vs GPT-4o-edited


In [ ]:
def evaluate_binoculars(texts, labels, batch_size=8):

    scores = []

    print(f"Evaluating {len(texts)} samples...")

    for i in tqdm(range(0, len(texts), batch_size), desc="Binoculars Scoring"):
        batch_texts = texts[i:i+batch_size]

        for text in batch_texts:
            try:
                score = binoculars.compute_score(text)
                if np.isnan(score) or np.isinf(score):
                    scores.append(np.nan)
                else:
                    scores.append(score)

            except Exception as e:
                print(f"\nError: {str(e)[:100]}")
                scores.append(np.nan)

    valid_scores = [s for s in scores if not np.isnan(s)]
    median_score = np.median(valid_scores) if valid_scores else 0.9
    scores = [median_score if np.isnan(s) else s for s in scores]

    print(f"Valid scores: {len(valid_scores)}/{len(texts)}")

    threshold = np.median(scores)
    predictions = [1 if s < threshold else 0 for s in scores]

    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    auroc = roc_auc_score(labels, [-s for s in scores])

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )

    metrics = {
        'accuracy': accuracy,
        'auroc': auroc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'threshold': threshold,
        'n_samples': len(texts),
        'valid_samples': len(valid_scores)
    }

    return scores, predictions, metrics

# Run evaluation on all 7 tasks
results_summary = {}

print("\n Starting evaluation \n")

for task_name, (col1, col2, description) in comparison_tasks.items():
    print(f"\n{'='*60}")
    print(f" Task: {task_name} - {description}")
    print(f"{'='*60}")

    df_task = df.copy()

    # Prepare data
    texts, labels = prepare_comparison_data(df_task, col1, col2, label1=0, label2=1)

    print(f" Total samples: {len(texts)}")
    print(f"   Class 0 ({col1}): {labels.count(0)}")
    print(f"   Class 1 ({col2}): {labels.count(1)}")

    # Evaluate
    scores, predictions, metrics = evaluate_binoculars(texts, labels)

    # Store results
    results_summary[task_name] = {
        'description': description,
        **metrics
    }

    # Save detailed results
    results_df = pd.DataFrame({
        'text': texts,
        'true_label': labels,
        'binoculars_score': scores,
        'prediction': predictions
    })
    results_df.to_csv(
        os.path.join(DIRS['results'], f'{task_name}_detailed.csv'),
        index=False
    )

    # Print metrics
    print(f"\n Results:")
    print(f"   • Samples:   {metrics['n_samples']} (valid: {metrics['valid_samples']})")
    print(f"   • Accuracy:  {metrics['accuracy']:.4f}")
    print(f"   • AUROC:     {metrics['auroc']:.4f}")
    print(f"   • Precision: {metrics['precision']:.4f}")
    print(f"   • Recall:    {metrics['recall']:.4f}")
    print(f"   • F1 Score:  {metrics['f1']:.4f}")


# Save Summary Results

summary_df = pd.DataFrame(results_summary).T
summary_df.to_csv(os.path.join(DIRS['results'], 'binoculars_summary.csv'))

print("\n" + "="*60)
print(" FINAL SUMMARY - All 7 Tasks")
print("="*60)
print(summary_df.to_string())

print(f"\n All results saved to {DIRS['results']}/")
print("\n Evaluation complete!")


 Starting evaluation 


 Task: H_vs_M - Human vs Machine-generated
 Total samples: 4374
   Class 0 (human_output): 2187
   Class 1 (model_output): 2187
Evaluating 4374 samples...


Binoculars Scoring: 100%|██████████| 547/547 [11:06<00:00,  1.22s/it]


Valid scores: 4374/4374

 Results:
   • Samples:   4374 (valid: 4374)
   • Accuracy:  0.7741
   • AUROC:     0.8391
   • Precision: 0.7744
   • Recall:    0.7737
   • F1 Score:  0.7740

 Task: E_vs_M - Expert-edited vs Machine-generated
 Total samples: 4374
   Class 0 (human_edits): 2187
   Class 1 (model_output): 2187
Evaluating 4374 samples...


Binoculars Scoring: 100%|██████████| 547/547 [11:34<00:00,  1.27s/it]


Valid scores: 4373/4374

 Results:
   • Samples:   4374 (valid: 4373)
   • Accuracy:  0.7003
   • AUROC:     0.7661
   • Precision: 0.7004
   • Recall:    0.7000
   • F1 Score:  0.7002

 Task: H_vs_E - Human vs Expert-edited
 Total samples: 4374
   Class 0 (human_output): 2187
   Class 1 (human_edits): 2187
Evaluating 4374 samples...


...

Valid scores: 8748/8748

 Results:
   • Samples:   8748 (valid: 8748)
   • Accuracy:  0.6716
   • AUROC:     0.7987
   • Precision: 0.9216
   • Recall:    0.6144
   • F1 Score:  0.7373

 Task: G_vs_M - GPT-4o-edited vs Machine-generated
 Total samples: 8748
   Class 0 (gpt-4o_edits): 6561
   Class 1 (model_output): 2187
Evaluating 8748 samples...


Binoculars Scoring: 100%|██████████| 1094/1094 [19:41<00:00,  1.08s/it]


Valid scores: 8748/8748

 Results:
   • Samples:   8748 (valid: 8748)
   • Accuracy:  0.5410
   • AUROC:     0.5791
   • Precision: 0.2909
   • Recall:    0.5816
   • F1 Score:  0.3879

 Task: H_vs_G - Human vs GPT-4o-edited
 Total samples: 8748
   Class 0 (human_output): 2187
   Class 1 (gpt-4o_edits): 6561
Evaluating 8748 samples...


Binoculars Scoring: 100%|██████████| 1094/1094 [18:43<00:00,  1.03s/it]


Valid scores: 8748/8748

 Results:
   • Samples:   8748 (valid: 8748)
   • Accuracy:  0.6669
   • AUROC:     0.7817
   • Precision: 0.9170
   • Recall:    0.6112
   • F1 Score:  0.7335

 FINAL SUMMARY - All 7 Tasks
                               description  accuracy     auroc precision    recall        f1 threshold n_samples valid_samples
H_vs_M          Human vs Machine-generated   0.77412  0.839096  0.774371  0.773663  0.774016  0.852071      4374          4374
E_vs_M  Expert-edited vs Machine-generated  0.700274  0.766148  0.700366  0.700046  0.700206  0.822222      4374          4373
H_vs_E              Human vs Expert-edited  0.593736  0.616434  0.593951  0.592593  0.593271  0.905063      4374          4373
L_vs_M   Llama-edited vs Machine-generated  0.548125  0.598936  0.297291  0.592135  0.395843  0.777778      8748          8748
H_vs_L               Llama-edited vs Human  0.671582  0.798671  0.921582  0.614388  0.737266  0.818879      8748          8748
G_vs_M  GPT-4o-edited v